# 06 — SFT Exploration

Explores the SFT pipeline — dataset analysis, chat template formatting, training curves, and before/after generation comparison.

Run after `make prepare-sft` and `make sft`.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import math
import random
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import torch

DATA_DIR = Path('../data')
RESULTS_DIR = Path('../results')
SFT_DIR = DATA_DIR / 'sft'

rng = random.Random(42)

## 1. Dataset overview

In [ ]:
def count_jsonl(path):
    if not path.exists():
        return 0
    with open(path) as f:
        return sum(1 for _ in f)

stages = {
    'chat/train': SFT_DIR / 'chat' / 'train.jsonl',
    'chat/val':   SFT_DIR / 'chat' / 'val.jsonl',
    'code/train': SFT_DIR / 'code' / 'train.jsonl',
    'code/val':   SFT_DIR / 'code' / 'val.jsonl',
}

print(f"{'Split':<15}  {'Examples':>10}  {'Path'}")
print("-" * 60)
for name, path in stages.items():
    count = count_jsonl(path)
    exists = '✓' if path.exists() else '✗ (run prepare-sft)'
    print(f"  {name:<13}  {count:>10,}  {exists}")

## 2. Chat dataset exploration

In [ ]:
chat_train = SFT_DIR / 'chat' / 'train.jsonl'

if chat_train.exists():
    chat_records = []
    with open(chat_train) as f:
        for i, line in enumerate(f):
            if i >= 5000:
                break
            chat_records.append(json.loads(line))

    # Text length distribution
    text_lengths = [len(r['text']) for r in chat_records]
    n_turns = [len(r['conversations']) for r in chat_records]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(text_lengths, bins=50, color='#4C72B0', alpha=0.85, edgecolor='white')
    axes[0].axvline(np.median(text_lengths), color='red', linestyle='--',
                    label=f'Median: {np.median(text_lengths):.0f}')
    axes[0].set_xlabel('Characters')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Chat SFT — Text Length Distribution', fontsize=11, fontweight='bold')
    axes[0].legend()
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

    turn_counts = Counter(n_turns)
    axes[1].bar(turn_counts.keys(), turn_counts.values(), color='#DD8452', alpha=0.85)
    axes[1].set_xlabel('Number of turns (messages)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Chat SFT — Conversation Length', fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print(f"Text length — mean: {np.mean(text_lengths):.0f}, median: {np.median(text_lengths):.0f}, p95: {np.percentile(text_lengths, 95):.0f}")
    print(f"Turns — mean: {np.mean(n_turns):.1f}, max: {max(n_turns)}")
else:
    print("Chat data not found — run: python finetune/data/prepare_sft.py --stage chat")

## 3. Sample chat conversations

In [ ]:
if chat_train.exists():
    samples = rng.sample(chat_records, min(3, len(chat_records)))
    for i, record in enumerate(samples):
        print(f"{'='*65}")
        print(f"Example {i+1} | Source: {record.get('source', 'N/A')} | Turns: {len(record['conversations'])}")
        print(f"{'='*65}")
        for msg in record['conversations']:
            role = msg['role'].upper()
            content = msg['content'][:300]
            print(f"[{role}]: {content}{'...' if len(msg['content']) > 300 else ''}")
        print()

## 4. Code dataset exploration

In [ ]:
code_train = SFT_DIR / 'code' / 'train.jsonl'

if code_train.exists():
    code_records = []
    with open(code_train) as f:
        for i, line in enumerate(f):
            if i >= 2000:
                break
            code_records.append(json.loads(line))

    code_lengths = [len(r['text']) for r in code_records]

    # Compare chat vs code length distributions
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(text_lengths if chat_train.exists() else [], bins=50, alpha=0.6,
            color='#4C72B0', label=f'Chat (n={len(text_lengths):,})', edgecolor='white')
    ax.hist(code_lengths, bins=50, alpha=0.6,
            color='#55A868', label=f'Code (n={len(code_lengths):,})', edgecolor='white')
    ax.set_xlabel('Characters')
    ax.set_ylabel('Count')
    ax.set_title('Text Length — Chat vs Code SFT', fontsize=12, fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Sample code example
    sample = rng.choice(code_records)
    print(f"{'='*65}")
    print("Sample Code Example")
    print(f"{'='*65}")
    for msg in sample['conversations']:
        role = msg['role'].upper()
        content = msg['content'][:400]
        print(f"[{role}]: {content}{'...' if len(msg['content']) > 400 else ''}")
else:
    print("Code data not found — run: python finetune/data/prepare_sft.py --stage code")

## 5. Chat template visualization

In [ ]:
# Show exactly how a conversation looks after formatting
if chat_train.exists():
    sample = rng.choice(chat_records)
    text = sample['text']

    # Highlight special tokens
    import re
    special_tokens = ['<|system|>', '<|user|>', '<|assistant|>', '<|endofturn|>',
                      '<|code|>', '<|endofcode|>']

    print("Formatted text (first 800 chars):")
    print("-" * 65)
    print(text[:800])
    print("..." if len(text) > 800 else "")
    print()

    # Token count
    tokenizer_path = DATA_DIR / 'tokenizer' / 'slm_tokenizer.json'
    if tokenizer_path.exists():
        from tokenizers import Tokenizer
        tokenizer = Tokenizer.from_file(str(tokenizer_path))
        n_tokens = len(tokenizer.encode(text).ids)
        print(f"Token count: {n_tokens:,}")
        print(f"Max seq len: 2048")
        print(f"Fits in one window: {'✓' if n_tokens <= 2048 else '✗ (will be truncated)'}")

## 6. SFT training curves

In [ ]:
def load_trainer_state(model_dir):
    state_files = list(model_dir.glob('**/trainer_state.json'))
    if not state_files:
        return None
    with open(sorted(state_files)[-1]) as f:
        return json.load(f)

chat_state = load_trainer_state(RESULTS_DIR / 'slm-125m-chat')
code_state = load_trainer_state(RESULTS_DIR / 'slm-125m-chat-code')

if chat_state or code_state:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    for ax, state, name, color in [
        (axes[0], chat_state, 'Chat SFT', '#4C72B0'),
        (axes[1], code_state, 'Code SFT', '#55A868'),
    ]:
        if not state:
            ax.text(0.5, 0.5, f'{name}\nnot trained yet',
                    ha='center', va='center', transform=ax.transAxes)
            continue

        logs = state.get('log_history', [])
        train_logs = [x for x in logs if 'loss' in x and 'eval_loss' not in x]
        eval_logs = [x for x in logs if 'eval_loss' in x]

        if train_logs:
            steps = [x['step'] for x in train_logs]
            losses = [x['loss'] for x in train_logs]
            ax.plot(steps, losses, alpha=0.4, color=color, linewidth=0.8, label='train')
            if len(losses) > 10:
                smoothed = np.convolve(losses, np.ones(10)/10, mode='valid')
                ax.plot(steps[5:-4], smoothed, color=color, linewidth=2)

        if eval_logs:
            eval_steps = [x['step'] for x in eval_logs]
            eval_losses = [x['eval_loss'] for x in eval_logs]
            ax.plot(eval_steps, eval_losses, color='#C44E52', linewidth=2,
                    marker='o', markersize=4, label='eval')

        ax.set_xlabel('Step')
        ax.set_ylabel('Loss')
        ax.set_title(name, fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)

    fig.suptitle('SFT Training Curves', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No SFT checkpoints found — run: make sft")

## 7. Before vs after SFT — generation comparison

In [ ]:
pretrain_dir = RESULTS_DIR / 'slm-125m' / 'final'
sft_dir = RESULTS_DIR / 'slm-125m-chat' / 'final'

if pretrain_dir.exists() and sft_dir.exists():
    from transformers import AutoConfig
    from model import SLMConfig, SLMForCausalLM
    from tokenizers import Tokenizer

    AutoConfig.register('slm', SLMConfig)
    tokenizer = Tokenizer.from_file(str(DATA_DIR / 'tokenizer' / 'slm_tokenizer.json'))

    base_model = SLMForCausalLM.from_pretrained(str(pretrain_dir)).eval()
    sft_model = SLMForCausalLM.from_pretrained(str(sft_dir)).eval()

    def generate(model, prompt, max_new_tokens=100):
        input_ids = torch.tensor([tokenizer.encode(prompt).ids])
        with torch.no_grad():
            output = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=0,
                eos_token_id=3,
            )
        return tokenizer.decode(output[0].tolist(), skip_special_tokens=False)

    # Format a chat prompt
    chat_prompt = "<|system|>You are a helpful assistant.<|endofturn|><|user|>Explain what a neural network is in simple terms.<|endofturn|><|assistant|>"

    print("PROMPT:")
    print(chat_prompt)
    print()
    print(f"{'='*65}")
    print("BASE MODEL (pretrained only):")
    print(generate(base_model, chat_prompt))
    print()
    print(f"{'='*65}")
    print("CHAT SFT MODEL:")
    print(generate(sft_model, chat_prompt))
else:
    print("Models not found — run: make pretrain && make sft")

## 8. Token budget analysis

In [ ]:
if chat_train.exists():
    tokenizer_path = DATA_DIR / 'tokenizer' / 'slm_tokenizer.json'

    if tokenizer_path.exists():
        from tokenizers import Tokenizer
        tokenizer = Tokenizer.from_file(str(tokenizer_path))

        chat_sample = chat_records[:500]
        token_counts = [len(tokenizer.encode(r['text']).ids) for r in chat_sample]

        over_2k = sum(1 for t in token_counts if t > 2048)
        over_4k = sum(1 for t in token_counts if t > 4096)

        print(f"Chat SFT token analysis (sample n={len(chat_sample)}):")
        print(f"  Mean tokens:     {np.mean(token_counts):.0f}")
        print(f"  Median tokens:   {np.median(token_counts):.0f}")
        print(f"  p95 tokens:      {np.percentile(token_counts, 95):.0f}")
        print(f"  > 2048 tokens:   {over_2k} ({100*over_2k/len(chat_sample):.1f}%) — will be truncated")
        print(f"  > 4096 tokens:   {over_4k} ({100*over_4k/len(chat_sample):.1f}%)")

        fig, ax = plt.subplots(figsize=(9, 4))
        ax.hist(token_counts, bins=40, color='#4C72B0', alpha=0.85, edgecolor='white')
        ax.axvline(2048, color='red', linestyle='--', label='max_seq_length=2048')
        ax.set_xlabel('Tokens')
        ax.set_ylabel('Count')
        ax.set_title('Chat SFT — Token Count Distribution', fontsize=11, fontweight='bold')
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("Tokenizer not found — run tokenizer training first")